   

# Declarative Automation Bundles: Infrastructure as Code

Welcome to the **Declarative Automation Bundles** (formerly Databricks Asset Bundles) demo! This notebook teaches you how to use bundles to manage your Databricks projects with software engineering best practices.

---

## What you'll learn:

* **Part 1:** Understanding Declarative Automation Bundles concepts
* **Part 2:** Bundle structure and configuration
* **Part 3:** Creating your first bundle
* **Part 4:** Deploying and managing bundles
* **Part 5:** CI/CD with bundles
* **Part 6:** Best practices and patterns

---

## Prerequisites:

* Databricks CLI v0.240+ installed locally
* Git installed (for version control)
* Basic understanding of YAML
* Familiarity with Lakeflow Jobs and Lakeflow Pipelines
* Completed previous Lakeflow demos (recommended)

---

**Let's get started!**

   

## Part 1: Understanding Declarative Automation Bundles

Before creating bundles, let's understand what they are and why they matter.

---

   

### What are Declarative Automation Bundles?

**Definition:**
* **Infrastructure as Code (IaC)** for Databricks
* Formerly known as **Databricks Asset Bundles (DABs)**
* Package your entire project as a deployable unit
* Include notebooks, Lakeflow Jobs, Lakeflow Pipelines, model serving endpoints, and configurations
* Enable CI/CD and software engineering best practices
* Version control everything as source files

---

### **The Problem Without Bundles:**

**Manual management:**
* Click through UI to create jobs and pipelines
* Hard to track changes over time
* Difficult to promote from dev to prod
* No code review for infrastructure changes
* Inconsistent configurations across environments
* Manual deployment process

---

### **The Solution With Bundles:**

**Automated, version-controlled infrastructure:**
* Define everything in code (YAML files)
* Version control with Git
* Code review for all changes
* Automated deployment with CI/CD
* Consistent environments (dev, staging, prod)
* Easy rollback and disaster recovery

---

### **Key Benefits:**

* **Version Control** - Track all changes in Git
* **Reproducibility** - Recreate environments easily
* **CI/CD** - Automated testing and deployment
* **Collaboration** - Code review and team workflows
* **Consistency** - Same structure across projects
* **Governance** - Audit trail and compliance
* **Unified Asset Management** - Deploy code, jobs, and infrastructure as a single unit

   

### What's Inside a Bundle?

A Declarative Automation Bundle contains:

---

### **1. Configuration Files (YAML)**

**databricks.yml:**
* Main configuration file (required)
* Defines bundle name, targets, and variables
* References other configuration files via `include`

**Resource files:**
* Lakeflow Job definitions
* Lakeflow Pipeline (formerly DLT) definitions
* Model serving endpoints
* MLflow Experiments and registered models
* Dashboards
* Databricks Apps

**Example structure:**
```
my_project/
├── databricks.yml          # Main config
├── resources/
│   ├── etl_job.yml         # Job definition
│   └── data_pipeline.yml   # Pipeline definition
└── src/
    └── notebooks/
        └── transform.py    # Source code
```

---

### **2. Source Code**

**Notebooks:**
* Python, SQL, Scala, R notebooks
* Business logic and transformations

**Python packages:**
* Custom libraries and modules
* Reusable functions and wheel packages

**SQL files:**
* Queries and DDL statements

---

### **3. Targets (Environments)**

**What they are:**
* Different deployment environments
* Each target has its own configuration overrides

**Common targets:**
* `dev` - Development environment
* `staging` - Pre-production testing
* `prod` - Production environment

**Example:**
```yaml
targets:
  dev:
    default: true
    mode: development
  
  prod:
    mode: production
    workspace:
      host: https://prod.cloud.databricks.com
    run_as:
      service_principal_name: prod-sp
```

---

### **4. Variables**

**What they are:**
* Parameterized values referenced with `${var.name}` syntax
* Different per environment via target overrides
* Reusable across resources

**Example:**
```yaml
variables:
  catalog:
    description: Unity Catalog name
    default: dev_catalog
  
  schema:
    description: Schema name
    default: bronze

targets:
  prod:
    variables:
      catalog: prod_catalog
      schema: gold
```

   

### Bundle Development Workflow

**Typical workflow with bundles:**

---

### **Step 1: Initialize**

```bash
# Create a new bundle from template
databricks bundle init
```

**What happens:**
* Presents a choice of default bundle templates
* Asks a series of questions to initialize project variables
* Creates project structure with configuration files

---

### **Step 2: Develop**

```bash
# Edit configuration files
vim databricks.yml
vim resources/my_job.yml

# Write source code
vim src/notebooks/transform.py
```

**What you do:**
* Define jobs, pipelines, and other resources in YAML
* Write notebooks and code
* Configure variables and settings per target

---

### **Step 3: Validate**

```bash
# Check configuration is valid
databricks bundle validate
```

**What happens:**
* Validates YAML syntax
* Checks resource definitions and references
* Returns a summary of the bundle identity and confirmation

---

### **Step 4: Deploy**

```bash
# Deploy to dev environment (default target)
databricks bundle deploy

# Deploy to production
databricks bundle deploy -t prod
```

**What happens:**
* Uploads source files to workspace
* Creates/updates jobs, pipelines, and other resources
* Applies configurations and permissions

---

### **Step 5: Run**

```bash
# Run a job from the bundle
databricks bundle run my_job -t dev
```

**What happens:**
* Triggers job execution in the target workspace
* Uses deployed configuration
* Returns run status and URL

---

### **Step 6: Iterate**

```bash
# Make changes
vim resources/my_job.yml

# Validate and redeploy
databricks bundle validate
databricks bundle deploy -t dev
```

**Continuous improvement:**
* Update configurations
* Add new resources
* Use `databricks bundle generate` to auto-generate config from existing workspace resources
* Use `databricks bundle deployment bind` to link config to existing resources

   

## Part 2: Bundle Structure and Configuration

Let's dive deep into bundle structure and configuration files.

---

   

### Recommended Project Structure

**Standard bundle layout:**

```
my_project/
├── databricks.yml              # Main bundle configuration
├── README.md                   # Project documentation
├── .gitignore                  # Git ignore file
│
├── resources/                  # Resource definitions (YAML)
│   ├── etl_job.yml             # Lakeflow Job config
│   ├── ml_training_job.yml     # ML training job config
│   └── data_pipeline.yml       # Lakeflow Pipeline config
│
├── src/                        # Source code
│   ├── notebooks/
│   │   ├── bronze_ingestion.py
│   │   ├── silver_transform.py
│   │   └── gold_aggregation.py
│   ├── pipelines/
│   │   └── dlt_pipeline.py     # Lakeflow Pipeline code
│   └── python/
│       └── utils/
│           └── helpers.py      # Shared utilities
│
├── tests/                      # Unit and integration tests
│   ├── unit/
│   │   └── test_transforms.py
│   └── integration/
│       └── test_pipeline.py
│
├── fixtures/                   # Test data and fixtures
│   └── sample_data.csv
│
└── .databricks/                # CLI state (auto-generated)
    └── bundle/
```

---

### **Key Directories:**

**`resources/`:**
* Contains all resource definitions
* YAML configuration files
* Tip: Use `include` globs in `databricks.yml` to auto-discover these files

**`src/`:**
* Source code for notebooks and libraries
* Business logic and transformations
* Reusable Python modules and wheel packages

**`tests/`:**
* Unit tests for functions (pytest)
* Integration tests for workflows (Databricks SDK)
* Test fixtures and data

---

### **Naming Conventions:**

**Resource files:**
* `<resource_name>.yml` - Keep it simple and descriptive
* Group by purpose rather than type for smaller projects
* For larger projects, use subdirectories: `resources/jobs/`, `resources/pipelines/`

**Source files:**
* Use descriptive names reflecting the medallion layer or function
* Follow Python naming conventions (snake\_case)
* Group by functionality or pipeline stage

   

### databricks.yml Configuration

**The main configuration file:**

---

### **Minimal Configuration:**

```yaml
bundle:
  name: my_project

targets:
  dev:
    default: true
```

---

### **Complete Configuration:**

```yaml
# Bundle identity
bundle:
  name: customer_analytics

# Include other config files (supports globs)
include:
  - resources/*.yml

# Define variables
variables:
  catalog:
    description: Unity Catalog name
    default: dev_catalog
  
  schema:
    description: Schema name  
    default: analytics
  
  environment:
    description: Environment name
    default: development

# Deployment targets
targets:
  # Development target
  dev:
    mode: development
    default: true
    variables:
      catalog: dev_catalog
      schema: dev_analytics
      environment: development
  
  # Staging target
  staging:
    variables:
      catalog: staging_catalog
      schema: staging_analytics
      environment: staging
  
  # Production target
  prod:
    mode: production
    workspace:
      root_path: /Shared/.bundle/prod/${bundle.name}
    
    # Run as service principal in prod
    run_as:
      service_principal_name: prod-sp-analytics
    
    variables:
      catalog: prod_catalog
      schema: prod_analytics
      environment: production
    
    # Permissions for production
    permissions:
      - level: CAN_VIEW
        group_name: data-analysts
      - level: CAN_MANAGE
        group_name: data-engineers
```

---

### **Key Sections:**

**`bundle:`**
* Bundle name (required, must be unique)

**`include:`**
* Reference other YAML files
* Supports glob patterns (e.g., `resources/**/*.yml`)
* Modularize configuration for larger projects

**`variables:`**
* Define reusable, parameterized values
* Override per target
* Reference with `${var.name}` syntax

**`targets:`**
* Define deployment environments
* Each has its own configuration overrides
* One marked as `default: true`
* `mode: development` for dev, `mode: production` for prod

**`run_as:`**
* Identity for resource execution
* User or service principal
* Required for `mode: production`

**Split for reuse:** For better modularity, split into separate files:
```yaml
# databricks.yml
bundle:
  name: hello-bundle
include:
  - '*.yml'

# hello-job.yml
resources:
  jobs:
    hello-job:
      name: hello-job
      tasks:
        - task_key: hello-task
          notebook_task:
            notebook_path: ./hello.py

# targets.yml
targets:
  dev:
    default: true
  prod:
    workspace:
      host: https://<workspace-host>
```

   

### Resource Configuration Files

**Defining Lakeflow Jobs, Lakeflow Pipelines, and other resources:**

---

### **Job Configuration Example:**

**File: `resources/etl_job.yml`**

```yaml
resources:
  jobs:
    customer_etl:
      name: "${bundle.target} - Customer ETL"
      
      tasks:
        - task_key: bronze_ingestion
          notebook_task:
            notebook_path: ../src/notebooks/bronze_ingestion.py
            base_parameters:
              catalog: ${var.catalog}
              schema: ${var.schema}
          # Use serverless compute (recommended)
          environment_key: default
        
        - task_key: silver_transform
          depends_on:
            - task_key: bronze_ingestion
          notebook_task:
            notebook_path: ../src/notebooks/silver_transform.py
            base_parameters:
              catalog: ${var.catalog}
              schema: ${var.schema}
          environment_key: default
      
      # Serverless environment definition
      environments:
        - environment_key: default
          spec:
            client: "1"
      
      schedule:
        quartz_cron_expression: "0 0 2 * * ?"
        timezone_id: "UTC"
        pause_status: UNPAUSED
      
      email_notifications:
        on_failure:
          - data-team@company.com
      
      tags:
        environment: ${var.environment}
        project: customer_analytics
```

---

### **Pipeline Configuration Example:**

**File: `resources/data_pipeline.yml`**

```yaml
resources:
  pipelines:
    customer_pipeline:
      name: "${bundle.target} - Customer Data Pipeline"
      
      catalog: ${var.catalog}
      target: ${var.schema}
      
      libraries:
        - notebook:
            path: ../src/pipelines/dlt_pipeline.py
      
      configuration:
        environment: ${var.environment}
      
      # Serverless compute (recommended)
      serverless: true
      
      development: true
      continuous: false
      channel: CURRENT
      
      notifications:
        - email_recipients:
            - data-team@company.com
          alerts:
            - on-update-failure
            - on-flow-failure
```

---

### **Variable References:**

**Use variables and substitutions in configurations:**

```yaml
# Reference bundle variables
${var.catalog}
${var.schema}
${var.environment}

# Reference bundle properties
${bundle.name}
${bundle.target}

# Reference workspace properties
${workspace.root_path}
${workspace.file_path}
```

---

### **Auto-generate from existing resources:**

```bash
# Generate bundle config from an existing job
databricks bundle generate job --existing-job-id 123456

# Generate bundle config from an existing pipeline
databricks bundle generate pipeline --existing-pipeline-id abc-def-123

# Bind generated config to the existing resource
databricks bundle deployment bind resources.jobs.customer_etl 123456
```

   

## Part 3: Creating Your First Bundle

Let's create a complete bundle from scratch.

---

   

### Prerequisites

**Before creating a bundle, ensure you have:**

---

### **1. Databricks CLI Installed**

**Check installation:**
```bash
databricks --version
# Requires v0.240.0 or later
```

**Install if needed:**
```bash
# Using Homebrew (macOS)
brew tap databricks/tap
brew install databricks

# Using curl (Linux/macOS)
curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh

# Using winget (Windows)
winget install Databricks.DatabricksCLI
```

> **Note:** The Databricks CLI is a standalone binary — do **not** install via `pip install databricks-cli` (that is the legacy CLI).

---

### **2. CLI Authentication**

**Recommended: OAuth (browser-based):**
```bash
databricks auth login --host https://your-workspace.cloud.databricks.com
```

**Or use a personal access token:**
```bash
databricks configure
```

**Verify authentication:**
```bash
databricks workspace list /
```

> **For CI/CD:** Databricks recommends **workload identity federation** for the most secure automated authentication. See Part 5.

---

### **3. Git Installed**

**Check installation:**
```bash
git --version
```

**Initialize repository:**
```bash
git init
git config user.name "Your Name"
git config user.email "your.email@company.com"
```

---

### **4. Text Editor or IDE**

**Recommended:**
* VS Code with the **Databricks extension for VS Code** (includes bundle support, deploy/run from IDE, Resource Explorer)
* PyCharm with Databricks Connect
* Any text editor with YAML support

> **New: Workspace bundles** — You can also create and deploy bundles directly from the Databricks workspace UI without any local tooling.

   

### Step 1: Initialize a New Bundle

**Create a bundle from template:**

---

### **Using the Interactive Template Chooser:**

```bash
# Navigate to your projects directory
cd ~/projects

# Initialize a new bundle (interactive)
databricks bundle init
```

**You'll be prompted to choose a template and answer setup questions.**

---

### **Available Default Templates:**

| Template | Description |
| --- | --- |
| **default-python** | Basic Python project with a Lakeflow Job and notebook task |
| **default-sql** | SQL project with warehouse-based tasks |
| **dbt-sql** | dbt project for SQL transformations |
| **mlops-stacks** | Production ML project with training, inference, and CI/CD |

---

### **For Lakeflow Pipelines:**

```bash
# Initialize a pipeline bundle specifically
databricks bundle init
# Then select the pipeline template

# Or use the pipelines init shortcut
databricks pipelines init
```

---

### **Custom Templates:**

You can create and use custom bundle templates for organizational consistency:
```bash
# Use a custom template from a Git repo
databricks bundle init https://github.com/your-org/bundle-template
```

---

### **What Gets Created (default-python):**

```
customer_analytics/
├── databricks.yml
├── README.md
├── resources/
│   └── customer_analytics_job.yml
└── src/
    └── notebook.py
```

   

### Step 2: Configure Your Bundle

**Customize the generated configuration:**

---

### **Edit databricks.yml:**

```yaml
# Bundle identity
bundle:
  name: customer_analytics

# Include resource files
include:
  - resources/*.yml

# Define variables
variables:
  catalog:
    description: Unity Catalog name
    default: dev_catalog
  
  schema:
    description: Schema name
    default: analytics

# Deployment targets
targets:
  # Development environment
  dev:
    mode: development
    default: true
    variables:
      catalog: dev_catalog
      schema: dev_analytics
  
  # Production environment
  prod:
    mode: production
    workspace:
      root_path: /Shared/.bundle/prod/${bundle.name}
    run_as:
      service_principal_name: prod-sp
    variables:
      catalog: prod_catalog
      schema: prod_analytics
```

---

### **Create Job Configuration:**

**File: `resources/etl_job.yml`**

```yaml
resources:
  jobs:
    customer_etl:
      name: "${bundle.target} - Customer ETL Job"
      
      tasks:
        - task_key: ingest_data
          notebook_task:
            notebook_path: ../src/ingest.py
            base_parameters:
              catalog: ${var.catalog}
              schema: ${var.schema}
          # Serverless compute
          environment_key: default
      
      environments:
        - environment_key: default
          spec:
            client: "1"
      
      schedule:
        quartz_cron_expression: "0 0 2 * * ?"
        timezone_id: "UTC"
      
      tags:
        project: customer_analytics
        environment: ${bundle.target}
```

---

### **Create Source Notebook:**

**File: `src/ingest.py`**

```python
# Databricks notebook source

# COMMAND ----------

# Get parameters
dbutils.widgets.text("catalog", "dev_catalog")
dbutils.widgets.text("schema", "analytics")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Using catalog: {catalog}")
print(f"Using schema: {schema}")

# COMMAND ----------

# Your ETL logic here
df = spark.read.table("samples.tpch.customer")

# Write to target
df.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.customers")

print(f"Data written to {catalog}.{schema}.customers")
```

   

### Step 3: Validate Configuration

**Check that your bundle is valid:**

---

### **Run Validation:**

```bash
# From bundle root directory
databricks bundle validate
```

**Successful output:**
```
Name: customer_analytics
Target: dev
Workspace:
  Host: https://your-workspace.cloud.databricks.com
  User: your.email@company.com
  Path: /Users/your.email@company.com/.bundle/customer_analytics/dev

Validation OK!
```

---

### **Common Validation Errors:**

**1. Invalid YAML syntax:**
```
Error: yaml: line 10: mapping values are not allowed in this context
```
**Fix:** Check indentation (use 2 spaces, not tabs) and ensure colons have spaces

**2. Missing required fields:**
```
Error: bundle.name is required
```
**Fix:** Add required field to `databricks.yml`

**3. Undefined variable reference:**
```
Error: variable "catalog" is not defined
```
**Fix:** Define the variable in the `variables:` section

**4. Invalid resource configuration:**
```
Error: jobs.customer_etl.tasks[0].notebook_task.notebook_path: file not found
```
**Fix:** Ensure notebook path is correct relative to the resource file location

---

### **Output Schema as JSON:**

```bash
# See the full configuration schema
databricks bundle schema

# View resolved config as JSON for debugging
databricks bundle validate --output json
```

**Use for reference when writing configurations.**

   

## Part 4: Deploying and Managing Bundles

Let's deploy and manage your bundle.

---

   

### Deploying Your Bundle

**Deploy to different environments:**

---

### **Deploy to Development:**

```bash
# Deploy to default target (dev)
databricks bundle deploy

# Or explicitly specify target
databricks bundle deploy -t dev
```

**What happens:**
1. Validates configuration
2. Uploads source files to workspace
3. Creates/updates Lakeflow Jobs, Pipelines, and other resources
4. Applies permissions and settings

**Output:**
```
Uploading customer_analytics to /Users/you/.bundle/customer_analytics/dev/files...
Deploying resources...
  Updated job: dev - Customer ETL Job

Deployment complete!
```

---

### **Deploy to Production:**

```bash
# Deploy to production target
databricks bundle deploy -t prod
```

**Important for production:**
* Uses `run_as` service principal
* Deploys to shared workspace location
* Applies production permissions
* Uses production variable overrides
* Requires `mode: production` which enforces stricter validation

---

### **Deployment Modes:**

**Development mode:**
```yaml
targets:
  dev:
    mode: development
```
* Prefixes resource names with `[dev <username>]` for isolation
* Deploys to personal workspace path
* Pauses all schedules and triggers by default
* Grants `CAN_MANAGE` and `IS_OWNER` to the deploying user

**Production mode:**
```yaml
targets:
  prod:
    mode: production
```
* Stricter validation (must not contain the current user's name in root\_path)
* Requires `run_as` or `permissions` configuration
* Resources are deployed as-is (no name prefixing)
* Schedules and triggers are active

---

### **View Deployed Resources:**

```bash
# List deployed resources and their workspace URLs
databricks bundle summary
```

**Output shows:**
* Jobs and their IDs/URLs
* Pipelines and their IDs/URLs
* Workspace paths
* Resource configuration summaries

   

### Running Bundle Resources

**Execute jobs and pipelines from your bundle:**

---

### **Run a Job:**

```bash
# Run job by resource key
databricks bundle run customer_etl -t dev
```

**What happens:**
* Deploys latest changes (if needed)
* Triggers the job in the target workspace
* Streams run output to terminal
* Returns final status and URL

---

### **Run a Pipeline:**

```bash
# Run pipeline by resource key
databricks bundle run customer_pipeline -t dev
```

---

### **Run with Parameters:**

```bash
# Override job parameters at runtime
databricks bundle run customer_etl --params catalog=test_catalog,schema=test_schema
```

---

### **Run from VS Code:**

With the Databricks extension for VS Code:
1. Open the **Bundle Resource Explorer** view
2. Select the job or pipeline resource
3. Click the **play** icon to deploy and run
4. View run status and click through to the workspace UI

---

### **Monitor Runs:**

```bash
# The run command streams output, or check in the UI
# The bundle summary also shows latest run status
databricks bundle summary -t dev
```

   

### Updating Your Bundle

**Make changes and redeploy:**

---

### **Workflow:**

**1. Make changes:**
```bash
# Edit configuration
vim resources/etl_job.yml

# Edit source code
vim src/ingest.py
```

**2. Validate changes:**
```bash
databricks bundle validate
```

**3. Deploy updates:**
```bash
databricks bundle deploy -t dev
```

**4. Test changes:**
```bash
databricks bundle run customer_etl -t dev
```

---

### **What Gets Updated:**

**Incremental updates:**
* Only changed files are uploaded
* Only modified resources are updated
* Existing runs are not affected

**Resource updates:**
* Jobs: Configuration updated, in-flight runs continue
* Pipelines: Configuration updated, may require refresh
* Notebooks: New version uploaded to workspace

---

### **Destroy Resources:**

```bash
# Remove all deployed resources for a target
databricks bundle destroy -t dev
```

**Warning:**
* Deletes all jobs, pipelines, and deployed resources
* Cannot be undone
* Use with caution, especially in production
* You will be prompted for confirmation

   

### Version Control with Git

**Track your bundle in Git:**

---

### **Initialize Git Repository:**

```bash
# Initialize Git
git init

# Create .gitignore
cat > .gitignore << EOF
# Databricks CLI state
.databricks/

# Python
__pycache__/
*.py[cod]
*$py.class
*.so
.Python
venv/
.venv/

# IDE
.vscode/
.idea/
*.swp
*.swo

# OS
.DS_Store
Thumbs.db
EOF
```

---

### **Commit Your Bundle:**

```bash
# Add files
git add .

# Commit
git commit -m "Initial bundle configuration"

# Create remote repository (GitHub, GitLab, etc.)
git remote add origin https://github.com/your-org/customer-analytics.git

# Push to remote
git push -u origin main
```

---

### **Branching Strategy:**

**Feature branches:**
```bash
# Create feature branch
git checkout -b feature/add-silver-layer

# Make changes
vim resources/etl_job.yml

# Commit changes
git add .
git commit -m "Add silver layer transformation"

# Push branch
git push origin feature/add-silver-layer
```

**Pull request workflow:**
1. Create feature branch
2. Make changes to bundle config and source code
3. Push and open pull request
4. CI validates bundle and runs tests
5. Code review
6. Merge to main triggers CD deployment

---

### **Source Control Recommendations:**

* **Small projects:** Single repository for both code and bundle configuration
* **Larger teams:** Consider separate repos for code and bundle config if release cycles differ
* Establish clear versioning contracts between repositories

---

### **What to Commit:**

**Do commit:**
* `databricks.yml`
* `resources/*.yml`
* `src/**/*`
* `tests/**/*`
* `README.md`
* `.gitignore`

**Don't commit:**
* `.databricks/` (auto-generated CLI state)
* Credentials or tokens
* Local environment files
* IDE-specific files

   

## Part 5: CI/CD with Declarative Automation Bundles

Automate testing and deployment with CI/CD pipelines.

---

   

### CI/CD Pipeline Overview

**Automated deployment workflow:**

---

### **Typical CI/CD Flow:**

```
┌─────────────────────────────────────────────────────────┐
│  Developer Workflow                                     │
├─────────────────────────────────────────────────────────┤
│  1. Create feature branch                               │
│  2. Make changes to bundle config + source code         │
│  3. Commit and push to Git                              │
│  4. Open pull request                                   │
└─────────────────────────────────────────────────────────┘
                        ↓
┌─────────────────────────────────────────────────────────┐
│  CI Pipeline (Automated on PR)                          │
├─────────────────────────────────────────────────────────┤
│  1. Checkout code                                       │
│  2. Install Databricks CLI (setup-cli action)           │
│  3. Authenticate via workload identity federation       │
│  4. Run unit tests (pytest)                             │
│  5. Validate bundle (databricks bundle validate)        │
│  6. Deploy to staging and run integration tests         │
└─────────────────────────────────────────────────────────┘
                        ↓
┌─────────────────────────────────────────────────────────┐
│  Code Review                                            │
├─────────────────────────────────────────────────────────┤
│  1. Review bundle config + source code changes          │
│  2. Check CI test results                               │
│  3. Approve or request changes                          │
└─────────────────────────────────────────────────────────┘
                        ↓
┌─────────────────────────────────────────────────────────┐
│  CD Pipeline (Automated on merge to main)               │
├─────────────────────────────────────────────────────────┤
│  1. Merge to main branch                                │
│  2. Deploy to production                                │
│  3. Run smoke tests                                     │
│  4. Send notifications                                  │
└─────────────────────────────────────────────────────────┘
```

---

### **Core CI/CD Principles:**

* **Version control everything** — Store notebooks, scripts, IaC definitions, and job configs in Git
* **Automate testing** — Unit tests with pytest, integration tests with Databricks SDK, validation with `bundle validate`
* **Employ Infrastructure as Code** — Define everything with Declarative Automation Bundles YAML (or Terraform)
* **Isolate environments** — Separate dev, staging, and production targets
* **Monitor and automate rollbacks** — Track deployment success rates and implement automated rollback
* **Unify asset management** — Deploy code, jobs, and infrastructure as a single unit with bundles

---

### **Authentication for CI/CD:**

> Databricks recommends **workload identity federation** for CI/CD authentication. It eliminates the need for Databricks secrets/tokens and is the most secure way to authenticate automated flows.

**Supported providers:**
* **AWS:** GitHub Actions with OIDC
* **Azure:** Azure DevOps with managed identity
* **GCP:** Cloud Build with workload identity

   

### GitHub Actions CI/CD

**Automate with GitHub Actions:**

---

### **File: `.github/workflows/deploy.yml`**

```yaml
name: Deploy Bundle

on:
  push:
    branches:
      - main
  pull_request:
    branches:
      - main

jobs:
  # Validate and test on pull requests
  validate:
    runs-on: ubuntu-latest
    if: github.event_name == 'pull_request'
    
    steps:
      - name: Checkout code
        uses: actions/checkout@v4
      
      - name: Install Databricks CLI
        uses: databricks/setup-cli@main
      
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      
      - name: Validate bundle
        run: databricks bundle validate -t dev
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
      
      - name: Run unit tests
        run: |
          pip install pytest
          pytest tests/unit/
  
  # Deploy to dev on pull request (for integration testing)
  deploy-dev:
    runs-on: ubuntu-latest
    needs: validate
    if: github.event_name == 'pull_request'
    
    steps:
      - name: Checkout code
        uses: actions/checkout@v4
      
      - name: Install Databricks CLI
        uses: databricks/setup-cli@main
      
      - name: Deploy to dev
        run: databricks bundle deploy -t dev
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
  
  # Deploy to production on merge to main
  deploy-prod:
    runs-on: ubuntu-latest
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'
    
    steps:
      - name: Checkout code
        uses: actions/checkout@v4
      
      - name: Install Databricks CLI
        uses: databricks/setup-cli@main
      
      - name: Deploy to production
        run: databricks bundle deploy -t prod
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
```

---

### **Authentication Setup:**

**Option 1: Service Principal (OAuth M2M) — Recommended:**

Add these secrets to your GitHub repository (Settings → Secrets and variables → Actions):
* `DATABRICKS_HOST` — Workspace URL (e.g., `https://your-workspace.cloud.databricks.com`)
* `DATABRICKS_CLIENT_ID` — Service principal application (client) ID
* `DATABRICKS_CLIENT_SECRET` — Service principal secret

**Option 2: Workload Identity Federation (OIDC) — Most Secure:**

Configure GitHub OIDC provider as a trusted identity in your Databricks account. No secrets needed in GitHub — authentication uses short-lived tokens.

> **Never use personal access tokens (PATs) in CI/CD.** Use service principals or workload identity federation.

---

### **Key Improvements Over Legacy Approach:**

* Uses `databricks/setup-cli@main` action (not `pip install databricks-cli`)
* Uses OAuth M2M or OIDC (not PATs stored in `~/.databrickscfg`)
* Uses `actions/checkout@v4` and `actions/setup-python@v5`
* Environment variables for auth (not config files)

   

### Testing Your Bundle

**Implement comprehensive testing:**

---

### **1. Unit Tests**

**Test individual functions:**

**File: `tests/unit/test_transforms.py`**

```python
import pytest
from pyspark.sql import SparkSession
from src.utils.transforms import clean_customer_data

@pytest.fixture
def spark():
    return SparkSession.builder.master("local[1]").getOrCreate()

def test_clean_customer_data(spark):
    # Create test data
    data = [
        (1, "John Doe", "john@email.com"),
        (2, "Jane Smith", "jane@email.com"),
        (3, None, "invalid@email.com")  # Invalid record
    ]
    df = spark.createDataFrame(data, ["id", "name", "email"])
    
    # Apply transformation
    result = clean_customer_data(df)
    
    # Assert results
    assert result.count() == 2  # Invalid record removed
    assert result.filter("name IS NULL").count() == 0
```

**Run tests:**
```bash
pytest tests/unit/
```

> **Tip:** Use `chispa` for Spark DataFrame assertions in tests.

---

### **2. Integration Tests**

**Test end-to-end workflows using the Databricks SDK:**

**File: `tests/integration/test_pipeline.py`**

```python
import pytest
from databricks.sdk import WorkspaceClient

def test_job_execution():
    w = WorkspaceClient()
    
    # Find the deployed job
    jobs = list(w.jobs.list(name="dev - Customer ETL Job"))
    assert len(jobs) > 0, "Job not found"
    
    job = jobs[0]
    
    # Run the job
    run = w.jobs.run_now(job_id=job.job_id)
    
    # Wait for completion
    result = w.jobs.wait_get_run_job_terminated_or_skipped(
        run_id=run.run_id
    )
    
    # Assert success
    assert result.state.result_state.value == "SUCCESS"
```

---

### **3. Bundle Validation Tests**

**Test configuration validity in CI:**

```bash
# Validate bundle config (runs in CI pipeline)
databricks bundle validate -t dev

# Returns non-zero exit code on failure, which fails CI
```

---

### **Run All Tests:**

```bash
# Run all tests
pytest tests/

# Run with coverage
pytest tests/ --cov=src --cov-report=html

# Run specific test file
pytest tests/unit/test_transforms.py -v
```

   

## Part 6: Best Practices and Patterns

Let's cover best practices for production bundles.

---

   

### Declarative Automation Bundles Best Practices

---

### **1. Project Organization**

**Use consistent structure:**

```
Good:
my_project/
├── databricks.yml
├── resources/
│   ├── etl_job.yml
│   └── data_pipeline.yml
├── src/
│   ├── notebooks/
│   ├── pipelines/
│   └── utils/
└── tests/

Bad:
my_project/
├── databricks.yml
├── job1.yml
├── job2.yml
├── notebook1.py
└── notebook2.py
```

**Benefits:**
* Easy to navigate
* Clear separation of concerns
* Scalable structure

---

### **2. Naming Conventions**

**Use descriptive names:**

```yaml
Good:
resources:
  jobs:
    customer_etl_daily:
      name: "${bundle.target} - Customer ETL Daily"
    
  pipelines:
    customer_bronze_to_gold:
      name: "${bundle.target} - Customer Data Pipeline"

Bad:
resources:
  jobs:
    job1:
      name: "job1"
```

**Include:**
* Environment prefix (`${bundle.target}`) for `mode: production` targets
* Purpose/function
* Frequency (if applicable)

> In `mode: development`, resource names are automatically prefixed with `[dev <username>]`.

---

### **3. Variable Management**

**Define reusable variables:**

```yaml
Good:
variables:
  catalog:
    description: Unity Catalog name
    default: dev_catalog
  
  schema:
    description: Schema name
    default: analytics
  
  notification_email:
    description: Email for notifications
    default: team@company.com

targets:
  prod:
    variables:
      catalog: prod_catalog
      notification_email: oncall@company.com

Bad:
# Hardcoded values in resources
resources:
  jobs:
    my_job:
      tasks:
        - notebook_task:
            base_parameters:
              catalog: dev_catalog  # Hardcoded!
```

---

### **4. Environment Separation**

**Use distinct targets with appropriate modes:**

```yaml
targets:
  dev:
    mode: development
    default: true
    variables:
      catalog: dev_catalog
  
  staging:
    variables:
      catalog: staging_catalog
  
  prod:
    mode: production
    run_as:
      service_principal_name: prod-sp
    variables:
      catalog: prod_catalog
```

---

### **5. Security Best Practices**

**Use service principals in production:**

```yaml
targets:
  prod:
    mode: production
    run_as:
      service_principal_name: prod-analytics-sp
    permissions:
      - level: CAN_VIEW
        group_name: data-analysts
      - level: CAN_MANAGE
        group_name: data-engineers
```

**Authentication hierarchy:**
* **CI/CD:** Workload identity federation (OIDC) or service principal OAuth
* **Production resources:** Service principal via `run_as`
* **Local dev:** OAuth browser login (`databricks auth login`)
* **Never:** Personal access tokens in CI/CD or hardcoded secrets

---

### **6. Resource Tagging**

**Tag all resources for governance and cost tracking:**

```yaml
resources:
  jobs:
    customer_etl:
      tags:
        environment: ${bundle.target}
        project: customer_analytics
        team: data-engineering
        cost_center: analytics
```

---

### **7. Use Bundle Templates for Consistency**

Use bundle templates to ensure consistent structure across teams:
* Use the default templates for quick starts
* Create **custom bundle templates** for your organization's standards
* Reduces setup errors and promotes best practices

---

### **8. Modularization**

**Split configuration into files:**

```yaml
Good:
databricks.yml:
  include:
    - resources/*.yml

resources/etl_job.yml
resources/ml_training_job.yml
resources/data_pipeline.yml

Bad:
databricks.yml:
  # Everything in one file (1000+ lines)
```

   

### Common Bundle Patterns

**Reusable patterns for common scenarios:**

---

### **Pattern 1: Multi-Environment Deployment**

**Use case:** Deploy same bundle to dev, staging, prod with different compute and data configurations

```yaml
variables:
  catalog:
    default: dev_catalog
  warehouse_size:
    description: SQL warehouse size
    default: Small

targets:
  dev:
    mode: development
    default: true
    variables:
      catalog: dev_catalog
      warehouse_size: Small
  
  staging:
    variables:
      catalog: staging_catalog
      warehouse_size: Medium
  
  prod:
    mode: production
    run_as:
      service_principal_name: prod-sp
    variables:
      catalog: prod_catalog
      warehouse_size: Large
```

---

### **Pattern 2: Serverless Compute**

**Use case:** Use serverless compute for jobs (recommended for most workloads)

```yaml
resources:
  jobs:
    etl_job:
      name: "${bundle.target} - ETL Job"
      tasks:
        - task_key: transform
          notebook_task:
            notebook_path: ../src/transform.py
          # Serverless compute
          environment_key: default
      
      environments:
        - environment_key: default
          spec:
            client: "1"
```

---

### **Pattern 3: Dynamic Resource Names**

**Use case:** Unique, identifiable names per environment

```yaml
resources:
  jobs:
    customer_etl:
      name: "[${bundle.target}] Customer ETL - ${bundle.name}"
  
  pipelines:
    data_pipeline:
      name: "[${bundle.target}] Data Pipeline - ${bundle.name}"
      catalog: ${var.catalog}
      target: ${var.schema}
```

**Results in:**
* Dev: `[dev] Customer ETL - customer_analytics`
* Prod: `[prod] Customer ETL - customer_analytics`

---

### **Pattern 4: Target-Specific Resources**

**Use case:** Deploy different resources per environment

```yaml
targets:
  dev:
    resources:
      jobs:
        # Dev-only job for testing
        test_job:
          name: "Test Job"
          tasks:
            - task_key: test
              notebook_task:
                notebook_path: ../src/test.py
  
  prod:
    resources:
      jobs:
        # Prod-only monitoring job
        monitoring_job:
          name: "Monitoring Job"
          schedule:
            quartz_cron_expression: "0 */5 * * * ?"
```

---

### **Pattern 5: Generating Config from Existing Resources**

**Use case:** Migrate existing workspace resources into a bundle

```bash
# Auto-generate job config from existing job
databricks bundle generate job --existing-job-id 123456

# Auto-generate pipeline config from existing pipeline
databricks bundle generate pipeline --existing-pipeline-id abc-def

# Bind the generated config to the existing resource
databricks bundle deployment bind resources.jobs.my_job 123456
```

This creates YAML config files and links them to the existing workspace resource so deployments update it in-place.

---

### **Pattern 6: Parameterized Notebooks**

**Use case:** Pass bundle variables to notebooks at runtime

```yaml
resources:
  jobs:
    etl:
      tasks:
        - task_key: transform
          notebook_task:
            notebook_path: ../src/etl.py
            base_parameters:
              catalog: ${var.catalog}
              schema: ${var.schema}
              environment: ${bundle.target}
```

**In notebook:**
```python
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("environment", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
environment = dbutils.widgets.get("environment")
```

   

### Troubleshooting Common Issues

**Solutions to common problems:**

---

### **Issue 1: Bundle Validation Fails**

**Error:**
```
Error: yaml: line 10: mapping values are not allowed in this context
```

**Solution:**
* Check YAML indentation (use 2 spaces, not tabs)
* Ensure colons have spaces after them
* Validate YAML syntax with an online linter

---

### **Issue 2: File Not Found**

**Error:**
```
Error: notebook_path: ../src/notebook.py: file not found
```

**Solution:**
* Check file path is relative to the resource file location
* Ensure file exists in the bundle project
* Use correct path separator (`/`)

---

### **Issue 3: Variable Not Defined**

**Error:**
```
Error: variable "catalog" is not defined
```

**Solution:**
```yaml
# Define variable in databricks.yml
variables:
  catalog:
    description: Catalog name
    default: dev_catalog
```

---

### **Issue 4: Permission Denied**

**Error:**
```
Error: User does not have permission to create job
```

**Solution:**
* Check workspace permissions
* Ensure user has `CAN_MANAGE` on workspace
* For production, use a service principal with appropriate permissions

---

### **Issue 5: Resource Already Exists (Name Conflict)**

**Error:**
```
Error: Job with name "Customer ETL" already exists
```

**Solution:**
* In `mode: development`, names are auto-prefixed — this shouldn't happen
* For production, use unique names per target: `"${bundle.target} - Customer ETL"`
* Use `databricks bundle deployment bind` to adopt an existing resource

---

### **Issue 6: Production Mode Validation Errors**

**Error:**
```
Error: production mode requires run_as or permissions to be set
```

**Solution:**
```yaml
targets:
  prod:
    mode: production
    run_as:
      service_principal_name: prod-sp
```

---

### **Debugging Tips:**

**1. Verbose/debug output:**
```bash
databricks bundle deploy -t dev --log-level debug
```

**2. View resolved configuration:**
```bash
databricks bundle validate -t dev --output json
```

**3. Check deployed resources:**
```bash
databricks bundle summary -t dev
```

**4. Check bundle schema for valid configuration options:**
```bash
databricks bundle schema
```

   

## Summary: Declarative Automation Bundles Mastery

Congratulations! You've learned how to use Declarative Automation Bundles (formerly Databricks Asset Bundles).

---

### **Key Concepts:**

**1. What are Bundles:**
* Infrastructure as Code for Databricks
* Package projects as deployable units
* Enable CI/CD and software engineering best practices
* Version control everything

**2. Bundle Structure:**
* `databricks.yml` — Main configuration
* `resources/` — Lakeflow Job and Pipeline definitions
* `src/` — Source code and notebooks
* `tests/` — Unit and integration tests

**3. Key Commands:**
* `databricks bundle init` — Create new bundle from template
* `databricks bundle validate` — Check configuration
* `databricks bundle deploy` — Deploy to environment
* `databricks bundle run` — Execute resources
* `databricks bundle destroy` — Remove resources
* `databricks bundle generate` — Auto-generate config from existing resources
* `databricks bundle deployment bind` — Link config to existing resources
* `databricks bundle summary` — View deployed resource details

**4. Targets:**
* Define multiple environments (dev, staging, prod)
* `mode: development` for dev (auto-prefixes names, pauses schedules)
* `mode: production` for prod (requires `run_as`, strict validation)
* Use variables for flexibility across targets

**5. CI/CD:**
* Automate testing and deployment
* Use GitHub Actions with `databricks/setup-cli` action
* Authenticate with workload identity federation or service principal OAuth
* Deploy on merge to main

---

### **Best Practices:**

* Use consistent project structure
* Define reusable variables (never hardcode)
* Separate environments with targets
* Use service principals in production (`run_as`)
* Tag all resources for governance
* Use bundle templates for organizational consistency
* Test before deploying (unit tests + `bundle validate`)
* Version control with Git (feature branch workflow)
* Implement CI/CD pipelines
* Modularize configuration with `include` globs

---

### **Next Steps:**

1. **Create your first bundle:**
   * Initialize with `databricks bundle init`
   * Configure for your project
   * Deploy to dev environment

2. **Set up CI/CD:**
   * Create GitHub repository
   * Add GitHub Actions workflow with `setup-cli`
   * Automate deployments with service principal auth

3. **Expand your bundle:**
   * Add more jobs and pipelines
   * Use `bundle generate` for existing resources
   * Implement testing and deploy to production

---

### **Resources:**

* [Declarative Automation Bundles Documentation](https://docs.databricks.com/en/dev-tools/bundles/)
* [Bundle Configuration Reference](https://docs.databricks.com/en/dev-tools/bundles/reference/)
* [Bundle Configuration Examples](https://docs.databricks.com/en/dev-tools/bundles/examples/)
* [CI/CD Best Practices](https://docs.databricks.com/en/dev-tools/ci-cd/best-practices/)
* [Databricks CLI Documentation](https://docs.databricks.com/en/dev-tools/cli/)
* [Bundle Examples GitHub Repository](https://github.com/databricks/bundle-examples)

   

## Hands-On Exercise: Create Your Bundle

**Follow these steps to create your first bundle:**

---

### **Step 1: Prerequisites**

```bash
# Install Databricks CLI
curl -fsSL https://raw.githubusercontent.com/databricks/setup-cli/main/install.sh | sh

# Authenticate
databricks auth login --host https://your-workspace.cloud.databricks.com

# Verify
databricks workspace list /
```

---

### **Step 2: Initialize Bundle**

```bash
# Create project directory
mkdir ~/projects/my_first_bundle
cd ~/projects/my_first_bundle

# Initialize bundle (interactive)
databricks bundle init

# When prompted:
# - Choose "default-python" template
# - Project name: my_first_bundle
```

---

### **Step 3: Explore Structure**

```bash
# View created files
tree .

# Should see:
# .
# ├── databricks.yml
# ├── README.md
# ├── resources/
# │   └── my_first_bundle_job.yml
# └── src/
#     └── notebook.py
```

---

### **Step 4: Customize Configuration**

**Edit `databricks.yml`:**

```yaml
bundle:
  name: my_first_bundle

include:
  - resources/*.yml

variables:
  catalog:
    default: main
  schema:
    default: default

targets:
  dev:
    mode: development
    default: true
```

---

### **Step 5: Validate**

```bash
# Validate configuration
databricks bundle validate

# Should see:
# Name: my_first_bundle
# Target: dev
# Validation OK!
```

---

### **Step 6: Deploy**

```bash
# Deploy to dev
databricks bundle deploy -t dev

# Should see:
# Uploading my_first_bundle to workspace...
# Deploying resources...
#   Updated job: [dev <username>] my_first_bundle_job
# Deployment complete!
```

---

### **Step 7: Run**

```bash
# Run the job
databricks bundle run my_first_bundle_job -t dev

# Monitor in terminal or follow the URL to the workspace UI
```

---

### **Step 8: Version Control**

```bash
# Initialize Git
git init

# Create .gitignore
echo ".databricks/" > .gitignore

# Commit
git add .
git commit -m "Initial bundle"
```

---

### **Step 9: Make Changes**

```bash
# Edit job configuration
vim resources/my_first_bundle_job.yml

# Validate changes
databricks bundle validate

# Redeploy
databricks bundle deploy -t dev
```

---

### **Step 10: Clean Up**

```bash
# Remove deployed resources
databricks bundle destroy -t dev
```

---

**Congratulations! You've created and deployed your first bundle!**

In [0]:
   

## Complete Example Bundle Files

**Copy these files to create a working bundle:**

---

### **File 1: databricks.yml**

```yaml
# Declarative Automation Bundle Configuration
# Main configuration file for the customer_analytics bundle

bundle:
  name: customer_analytics

# Include all resource definition files
include:
  - resources/*.yml

# Define reusable variables
variables:
  catalog:
    description: Unity Catalog name
    default: dev_catalog
  
  schema:
    description: Schema name within the catalog
    default: analytics
  
  notification_email:
    description: Email address for job notifications
    default: data-team@company.com

# Deployment targets
targets:
  # Development environment
  dev:
    mode: development
    default: true
    variables:
      catalog: dev_catalog
      schema: dev_analytics
  
  # Staging environment
  staging:
    variables:
      catalog: staging_catalog
      schema: staging_analytics
  
  # Production environment
  prod:
    mode: production
    workspace:
      root_path: /Shared/.bundle/prod/${bundle.name}
    
    run_as:
      service_principal_name: prod-analytics-sp
    
    variables:
      catalog: prod_catalog
      schema: prod_analytics
    
    permissions:
      - level: CAN_VIEW
        group_name: data-analysts
      - level: CAN_MANAGE
        group_name: data-engineers
```

---

### **File 2: resources/etl_job.yml**

```yaml
# Customer ETL Job Configuration
resources:
  jobs:
    customer_etl:
      name: "${bundle.target} - Customer ETL Job"
      
      tasks:
        - task_key: bronze_ingestion
          description: "Ingest raw customer data"
          notebook_task:
            notebook_path: ../src/notebooks/bronze_ingestion.py
            base_parameters:
              catalog: ${var.catalog}
              schema: ${var.schema}
          environment_key: default
        
        - task_key: silver_transform
          description: "Clean and transform data"
          depends_on:
            - task_key: bronze_ingestion
          notebook_task:
            notebook_path: ../src/notebooks/silver_transform.py
            base_parameters:
              catalog: ${var.catalog}
              schema: ${var.schema}
          environment_key: default
        
        - task_key: gold_aggregation
          description: "Create business aggregations"
          depends_on:
            - task_key: silver_transform
          notebook_task:
            notebook_path: ../src/notebooks/gold_aggregation.py
            base_parameters:
              catalog: ${var.catalog}
              schema: ${var.schema}
          environment_key: default
      
      # Serverless compute
      environments:
        - environment_key: default
          spec:
            client: "1"
      
      schedule:
        quartz_cron_expression: "0 0 2 * * ?"
        timezone_id: "UTC"
        pause_status: UNPAUSED
      
      email_notifications:
        on_failure:
          - ${var.notification_email}
      
      max_retries: 2
      timeout_seconds: 7200
      
      tags:
        project: customer_analytics
        team: data-engineering
```

---

### **File 3: resources/data_pipeline.yml**

```yaml
# Customer Data Pipeline Configuration (Lakeflow Pipeline)
resources:
  pipelines:
    customer_pipeline:
      name: "${bundle.target} - Customer Data Pipeline"
      
      catalog: ${var.catalog}
      target: ${var.schema}
      
      libraries:
        - notebook:
            path: ../src/pipelines/customer_dlt_pipeline.py
      
      configuration:
        environment: ${bundle.target}
      
      serverless: true
      development: true
      continuous: false
      channel: CURRENT
      
      notifications:
        - email_recipients:
            - ${var.notification_email}
          alerts:
            - on-update-failure
            - on-flow-failure
```

---

### **File 4: src/notebooks/bronze_ingestion.py**

```python
# Databricks notebook source
# MAGIC %md
# MAGIC # Bronze Layer: Customer Data Ingestion

# COMMAND ----------

dbutils.widgets.text("catalog", "dev_catalog")
dbutils.widgets.text("schema", "analytics")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Configuration:")
print(f"  Catalog: {catalog}")
print(f"  Schema: {schema}")

# COMMAND ----------

from pyspark.sql.functions import current_timestamp, lit

df = spark.read.table("samples.tpch.customer")

df_bronze = (df
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source", lit("tpch"))
)

df_bronze.write.mode("overwrite").saveAsTable(
    f"{catalog}.{schema}.customers_bronze"
)

print(f"Data written to {catalog}.{schema}.customers_bronze")
```

---

### **File 5: .github/workflows/deploy.yml**

```yaml
name: Deploy Bundle

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  validate-and-deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      
      - name: Install Databricks CLI
        uses: databricks/setup-cli@main
      
      - name: Validate bundle
        if: github.event_name == 'pull_request'
        run: databricks bundle validate -t dev
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
      
      - name: Deploy to production
        if: github.event_name == 'push' && github.ref == 'refs/heads/main'
        run: databricks bundle deploy -t prod
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
```